# Baseline: Zabka location prediction

Notebook baseline required by the project brief: EDA, preprocessing, baseline model and evaluation.

In [ ]:
import json
from pathlib import Path

import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import average_precision_score, roc_auc_score
from sklearn.model_selection import train_test_split

from zabki_prediction.pipelines.site_selection.nodes import parse_zabka_geojson, build_model_tables


In [ ]:
raw = json.loads(Path('../zabka_locs.geojson').read_text(encoding='utf-8'))
stores = parse_zabka_geojson(raw)
stores.head(), stores.shape

In [ ]:
stores[['lon', 'lat']].describe()

In [ ]:
params = {
    'random_state': 42,
    'poland_bbox': {'min_lon': 14.1, 'max_lon': 24.2, 'min_lat': 49.0, 'max_lat': 54.9},
    'candidate_grid_step_degrees': 0.10,
    'min_candidate_distance_km': 0.35,
    'negative_ratio': 1.0,
}
train, candidates = build_model_tables(stores, params)
train.head(), train['label'].value_counts()

In [ ]:
feature_cols = [c for c in train.columns if c not in ['candidate_id', 'label']]
X_train, X_valid, y_train, y_valid = train_test_split(
    train[feature_cols], train['label'], test_size=0.2, random_state=42, stratify=train['label']
)
baseline = RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1)
baseline.fit(X_train, y_train)
scores = baseline.predict_proba(X_valid)[:, 1]
{'roc_auc': roc_auc_score(y_valid, scores), 'average_precision': average_precision_score(y_valid, scores)}